In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Описание данных

**Целевая переменная:** `SeriousDlqin2yrs` — был ли у клиента просрочка 90+ дней или хуже за последние 2 года.

| Признак | Описание | Тип данных |
|---------|----------|-------------|
| `SeriousDlqin2yrs` | Просрочка 90+ дней или хуже за последние 2 года | Y/N (целевой) |
| `RevolvingUtilizationOfUnsecuredLines` | Отношение общего остатка по кредитным картам и персональным кредитным линиям (кроме недвижимости и рассрочки) к сумме кредитных лимитов | доля |
| `age` | Возраст заёмщика в годах | целое число |
| `NumberOfTime30-59DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 30–59 дней (но не более) за последние 2 года | целое число |
| `DebtRatio` | Отношение ежемесячных выплат по долгам, алиментов, расходов на жизнь к валовому месячному доходу | доля |
| `MonthlyIncome` | Ежемесячный доход | вещественное число |
| `NumberOfOpenCreditLinesAndLoans` | Количество открытых кредитов (например, автокредит, ипотека) и кредитных линий (например, кредитные карты) | целое число |
| `NumberOfTimes90DaysLate` | Количество раз, когда заёмщик допускал просрочку 90+ дней | целое число |
| `NumberRealEstateLoansOrLines` | Количество ипотечных кредитов и кредитов под недвижимость, включая возобновляемые кредитные линии под залог недвижимости | целое число |
| `NumberOfTime60-89DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 60–89 дней (но не более) за последние 2 года | целое число |
| `NumberOfDependents` | Количество иждивенцев в семье (не включая самого заёмщика) — супруг(а), дети и т.д. | целое число |


In [ ]:
train_file = Path('data.csv')
if not train_file.exists():
    train_file = Path('hw1/data.csv')

In [ ]:
df = pd.read_csv(train_file)
df.head()

In [ ]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

In [ ]:
feature_names = X.columns.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

## Задание

Обучить линейную модель предсказания целевой переменной. Оценить качество. Считается, что дать кредит человеку, который допустит просрочку, для банка в 5 раз дороже, чем не дать платёжеспособному.

Ноутбук должен быть рабочим. Также должна быть реализована функция predict(model, data_path), возвращающая предсказанные значения. 

In [ ]:
def predict(model, data_path):
    df = pd.read_csv(data_path)
    X_input = df.drop(columns=['SeriousDlqin2yrs'], errors='ignore')
    X_input = X_input.reindex(columns=model.feature_names_)

    proba = model.predict_proba(X_input)[:, 1]
    threshold = getattr(model, 'threshold_', 0.5)
    return (proba >= threshold).astype(int)


In [ ]:
def business_cost(y_true, y_pred, fn_cost=5, fp_cost=1):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = fn_cost * fn + fp_cost * fp
    avg_cost = total_cost / len(y_true)
    return total_cost, avg_cost, (tn, fp, fn, tp)

valid_proba = model.predict_proba(X_valid)[:, 1]
threshold_grid = np.linspace(0.01, 0.99, 199)
threshold_rows = []

for threshold in threshold_grid:
    valid_pred = (valid_proba >= threshold).astype(int)
    total_cost, avg_cost, _ = business_cost(y_valid, valid_pred)
    threshold_rows.append((threshold, total_cost, avg_cost))

threshold_df = pd.DataFrame(threshold_rows, columns=['threshold', 'total_cost', 'avg_cost'])
best_threshold = threshold_df.sort_values(['total_cost', 'threshold'], ascending=[True, True]).iloc[0]['threshold']

model.feature_names_ = feature_names
model.threshold_ = float(best_threshold)

# Accuracy здесь недостаточна: классы несбалансированы, и цена ошибок разная.
# Поэтому подбираем порог по функции стоимости 5 * FN + FP, а не фиксируем 0.5.
threshold_df.nsmallest(5, 'total_cost')

In [ ]:
import tempfile

test_proba = model.predict_proba(X_test)[:, 1]
test_pred_default = (test_proba >= 0.5).astype(int)
test_pred_best = (test_proba >= model.threshold_).astype(int)

default_total_cost, default_avg_cost, default_cm = business_cost(y_test, test_pred_default)
best_total_cost, best_avg_cost, best_cm = business_cost(y_test, test_pred_best)

print(f'Validation threshold: {model.threshold_:.3f}')
print(f'Accuracy at 0.5: {accuracy_score(y_test, test_pred_default):.4f}')
print(f'Accuracy at best threshold: {accuracy_score(y_test, test_pred_best):.4f}')
print(f'Precision at best threshold: {precision_score(y_test, test_pred_best, zero_division=0):.4f}')
print(f'Recall at best threshold: {recall_score(y_test, test_pred_best, zero_division=0):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, test_proba):.4f}')
print(f'At 0.5 -> FP: {default_cm[1]}, FN: {default_cm[2]}, total_cost: {default_total_cost}')
print(f'At best threshold -> FP: {best_cm[1]}, FN: {best_cm[2]}, total_cost: {best_total_cost}')
print(f'Business cost at best threshold: {best_avg_cost:.4f}')
print('Confusion matrix at best threshold [tn, fp; fn, tp]:')
print(np.array([[best_cm[0], best_cm[1]], [best_cm[2], best_cm[3]]]))

train_preds = predict(model, train_file)
print(f'Predictions for train file: {len(train_preds)} rows')

with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as tmp:
    predict_check_path = Path(tmp.name)

df.drop(columns=['SeriousDlqin2yrs']).to_csv(predict_check_path, index=False)
predict_without_target = predict(model, predict_check_path)
predict_check_path.unlink(missing_ok=True)

print(f'Predictions without target column: {len(predict_without_target)} rows')
print(f'Length check passed: {len(predict_without_target) == len(df)}')